# FraudLens: Notebook 02 — Feature Engineering & NLP Pipeline
### Construction of Multi-Modal Lexical, Stylometric, and Security Feature Spaces

In this notebook, we implement:
1. Lexical Preprocessing (Hinglish phonetic normalization, URL token masking, phone token masking)
2. Sublinear TF-IDF Vectorization with N-Grams $(1, 2)$
3. Custom Scikit-Learn `DomainFeatureExtractor` for dense security signals
4. Feature matrix composition via `FeatureUnion`


In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
from ai_engine.preprocessing.text_cleaner import clean_text
from ai_engine.features.feature_pipeline import build_feature_pipeline, DomainFeatureExtractor

df = pd.read_csv('../ai_engine/data/processed/combined_fraud_dataset.csv')
sample_texts = df['text'].head(5).tolist()

print("Original Text vs Cleaned Representation:")
for original in sample_texts[:3]:
    print(f"\nRAW: {original}")
    print(f"CLEANED: {clean_text(original)}")


## 1. Dense Domain Feature Extraction
We test the `DomainFeatureExtractor` which measures uppercase ratio, digit density, URL indicators, UPI VPA presence, and urgency scores.


In [ ]:
extractor = DomainFeatureExtractor()
dense_matrix = extractor.transform(df['text'].values)

feature_names = [
    "length", "word_count", "caps_ratio", "digit_density",
    "exclamation_count", "question_count", "has_url", "has_phone",
    "has_vpa", "urgency_score", "reverse_debit_score", "credential_score"
]

dense_df = pd.DataFrame(dense_matrix, columns=feature_names)
dense_df['target'] = df['target'].values
dense_df.groupby('target').mean().round(3).T
